# 🧠 Synthetic Instruction Dataset for LLM Fine-Tuning

In this lesson, we are going to generate a synthetic instruction dataset — question-and-answer pairs grounded in real documents, that can be used to fine-tune or evaluate a large language model without exposing the source content.

In this notebook, we will:
1. Generate Q&A pairs from a folder of documents using `DocumentQAGeneration`
2. Inspect and quality-filter the output

## 1. Configure the LLM Backend

`DocumentQAGeneration` uses an LLM to read each document and generate question-answer pairs from its content. The LLM is wired up through the environment — you can use Anthropic Claude or OpenAI.

In [ ]:
# Option A — Anthropic Claude (used in this notebook)
os.environ['ANTHROPIC_API_KEY'] = 'YOUR_ANTHROPIC_KEY_HERE'  # ← replace with your key

# Option B — OpenAI (uncomment to use instead)
# os.environ['OPENAI_API_KEY'] = 'YOUR_OPENAI_KEY_HERE'

print('LLM backend configured ✓')

## 2. Generate Q&A Pairs

`generate()` accepts a folder path, a single file path, or a PyArrow table as `input_source`. `num_qa_pairs` sets how many pairs to generate **per document**.

In [ ]:
from ydata.synthesizers.text.model.qa import DocumentQAGeneration

qa_generator = DocumentQAGeneration()

print('DocumentQAGeneration ready ✓')

In [ ]:
# Generate Q&A pairs from a folder of documents
# Replace 'assets/documents/' with the path to your own document folder
result = qa_generator.generate(
    input_source='assets/documents/',   # folder, single file, or PyArrow table
    docs_extension='docx',              # 'docx' | 'txt' | 'pdf'
    num_qa_pairs=5,                     # pairs per document
)

print(f'Generated {len(result)} Q&A pairs ✓')

## 3. Inspect the Output

In [ ]:
print(f'Shape: {result.shape}')
print(f'Columns: {list(result.columns)}')
result.head()

In [ ]:
# Distribution of Q&A pairs across source documents
print('Pairs per source document:')
print(result['source'].value_counts().to_string())

In [ ]:
# Spot-check a single Q&A pair
sample = result.sample(1).iloc[0]
print(f'Source  : {sample["source"]}')
print(f'Question: {sample["question"]}')
print(f'Answer  : {sample["answer"]}')

## 4. Quality Filtering

In [ ]:
MIN_ANSWER_WORDS = 5    # discard single-word or near-empty answers
MIN_QUESTION_WORDS = 3  # discard fragment questions

filtered = result[
    (result['answer'].str.split().str.len() >= MIN_ANSWER_WORDS) &
    (result['question'].str.split().str.len() >= MIN_QUESTION_WORDS)
].reset_index(drop=True)

print(f'Before filtering : {len(result):>4} pairs')
print(f'After filtering  : {len(filtered):>4} pairs')
print(f'Removed          : {len(result) - len(filtered):>4} pairs')

## 5. Export to JSONL for Fine-Tuning

Instruction fine-tuning pipelines expect data in JSONL format — one JSON object per line. `question` maps to `instruction` and `answer` maps to `response`.

In [ ]:
output_path = 'output/instruction_dataset.jsonl'
os.makedirs('output', exist_ok=True)

with open(output_path, 'w') as f:
    for _, row in filtered.iterrows():
        record = {
            'instruction': row['question'],
            'response':    row['answer'],
            'source':      row['source'],
        }
        f.write(json.dumps(record) + '\n')

print(f'Exported {len(filtered)} pairs → {output_path}')

In [ ]:
# Preview the first three records
with open(output_path) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(json.dumps(json.loads(line), indent=2))
        print()

> **⏸ Active Learning** — Pause here. Change `docs_extension` from `'docx'` to `'txt'` and re-run with the same folder. Compare the Q&A pairs: do they cover different aspects of the same documents? Then try increasing `num_qa_pairs` from 5 to 15 on a single document and check whether the extra pairs are genuinely diverse.

## ✅ Summary

In this lesson you:

1. **Configured an LLM backend** — wired an Anthropic or OpenAI key into the environment so `DocumentQAGeneration` can call it internally
2. **Generated Q&A pairs** using `DocumentQAGeneration.generate()`, pointed at a folder of documents with a specified file type and pairs-per-document count
3. **Inspected the output** — a DataFrame with `question`, `answer`, and `source` columns, traceable back to each source file

The key insight: synthetic instruction datasets are only as good as their source documents. Combining `DocumentGenerator` (from the previous video) with `DocumentQAGeneration` gives you a fully synthetic pipeline — generate the documents, then generate the instruction dataset from them.

> **Active Learning:** Try generating synthetic invoices with `DocumentGenerator` in DOCX format, then immediately feeding that output folder into `DocumentQAGeneration`. You now have an end-to-end synthetic pipeline — source documents and instruction dataset — with no real data anywhere in the chain.